In [1]:
# Instalações para Google Colab
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas openpyxl > /dev/null

# Imports
import os, re, time, requests, zipfile
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# Função para limpar texto
def limpar(texto):
    return ' '.join(texto.strip().split()) if texto else "N/A"

# Data e pastas
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_casa_rossi"
os.makedirs(pasta_base, exist_ok=True)

# WebDriver Headless
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# Função para coletar os links dos produtos, avançando pelas páginas
def coletar_links_produtos():
    # Usamos a URL base conforme exibido no HTML da paginação
    url_base = 'https://www.casarossi.com.br/pisos-e-revestimentos'
    headers = {"User-Agent": "Mozilla/5.0"}

    # Primeiro acesso para identificar o total de páginas
    response = requests.get(url_base, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    # O botão "Última página" contém o número máximo
    ultima_pagina_tag = soup.select_one("li.page.last a")
    total_paginas = int(re.search(r'pg=(\d+)', ultima_pagina_tag['href']).group(1)) if ultima_pagina_tag else 1
    print(f"Total de páginas encontradas: {total_paginas}")

    links = []
    # Itera por todas as páginas
    for pagina in range(1, total_paginas + 1):
        url_pagina = f"{url_base}?pg={pagina}"
        print(f"Processando a página: {url_pagina}")
        response = requests.get(url_pagina, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        produtos = soup.find_all('li', class_='item flex')
        for produto in produtos:
            a_tag = produto.find('a', class_='space-image')
            if a_tag:
                link = a_tag['href']
                if not link.startswith("http"):
                    link = "https://www.casarossi.com.br" + link
                links.append(link)
        time.sleep(1)
    return links

# Coleta dos links dos produtos avançando pelas páginas
links_produtos = coletar_links_produtos()
print(f"Total de produtos coletados: {len(links_produtos)}")

dados = []

# Para cada link coletado, usamos Selenium para extrair os dados detalhados
for i, url in enumerate(links_produtos):
    print(f"[{i+1}/{len(links_produtos)}] Coletando: {url}")
    try:
        driver.get(url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        nome = limpar(soup.find('h1', class_='product-name').text)
        codigo = limpar(soup.select_one('.product-sku .ref').text) if soup.select_one('.product-sku .ref') else nome[:20]
        preco_original_tag = soup.select_one('.PrecoPrincipal span')
        preco_caixa = limpar(preco_original_tag.text) if preco_original_tag else "N/A"
        preco_m2_tag = soup.select_one('.preco-por-metro')
        preco_m2 = limpar(preco_m2_tag.text.replace('por m²', '').replace('R$', '').strip()) if preco_m2_tag else "N/A"
        m2_caixa_tag = soup.select_one('.metro-na-caixa')
        m2_caixa = limpar(m2_caixa_tag.text.replace('Caixa com', '').replace('m²', '').strip()) if m2_caixa_tag else "N/A"

        # Imagem
        imagem_tag = soup.select_one('.zoom img')
        imagem_url = imagem_tag['src'] if imagem_tag else "N/A"

        # DESCRIÇÃO: Utiliza o conteúdo da DIV com id="descricao"
        descricao_html = soup.find('div', id='descricao')
        descricao = limpar(descricao_html.get_text(separator=' ')) if descricao_html else "N/A"

        # Especificações: Extração a partir da div com id="caracteristicas"
        espec_div = soup.find('div', id='caracteristicas')
        if espec_div:
            itens = espec_div.find_all('li')
            especificacoes = ' | '.join([limpar(li.text) for li in itens])
        else:
            especificacoes = "N/A"

        # Comparar preços
        preco_promocional = preco_caixa
        if preco_m2 != "N/A" and preco_caixa != "N/A" and preco_caixa == preco_caixa:
            preco_promocional = "N/A"

        dados.append([
            codigo,
            nome,
            preco_promocional,
            preco_caixa,
            preco_m2,
            especificacoes,
            descricao,
            imagem_url,
            url,
            "Casa Rossi"
        ])

    except Exception as e:
        print(f"Erro no produto: {e}")

driver.quit()

# Criar DataFrame com os dados coletados
df = pd.DataFrame(dados, columns=[
    "Código", "Nome", "Preço Promocional", "Preço Original", "Preço por m²",
    "Especificações", "Descrição", "Imagem", "URL", "Loja"
])

# Função para ajustar os valores numéricos de preço
def limpar_preco(valor):
    try:
        return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        return None

df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)
df["Preço Original"] = df["Preço Original"].apply(limpar_preco)
df["Preço Promocional"] = df.apply(
    lambda row: row["Preço Promocional"] if row["Preço Promocional"] != row["Preço Original"] else "N/A", axis=1
)

# Salvar CSV e Excel
csv_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.csv")
xlsx_path = os.path.join(pasta_base, f"casa_rossi_{data_hoje}.xlsx")

df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
df.to_excel(xlsx_path, index=False)

# Baixar imagens
images_dir = os.path.join(pasta_base, f"imagens_casa_rossi_{data_hoje}")
os.makedirs(images_dir, exist_ok=True)

for index, row in df.iterrows():
    codigo = row["Código"]
    img_url = row["Imagem"]
    if img_url != "N/A":
        try:
            resp = requests.get(img_url, stream=True, timeout=10)
            if resp.status_code == 200:
                imagem_path = os.path.join(images_dir, f"{codigo}.png")
                with open(imagem_path, "wb") as f:
                    for chunk in resp.iter_content(1024):
                        f.write(chunk)
            else:
                print(f"❌ Erro ao baixar imagem para o código {codigo}: Status code {resp.status_code}")
        except Exception as e:
            print(f"❌ Erro ao baixar imagem para o código {codigo}: {e}")
    else:
        print(f"❌ Nenhuma imagem encontrada para o código {codigo}.")

# Compactar as imagens em um arquivo ZIP
zip_filename = os.path.join(pasta_base, f"imagens_casa_rossi_{data_hoje}.zip")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for foldername, subfolders, filenames in os.walk(images_dir):
        for filename in filenames:
            file_path = os.path.join(foldername, filename)
            arcname = os.path.relpath(file_path, images_dir)
            zipf.write(file_path, arcname=arcname)

# Mensagens finais com ícones
print(f"📄 CSV salvo em: {csv_path}")
print(f"📊 Excel salvo em: {xlsx_path}")
print(f"📦 Imagens compactadas em: {zip_filename}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
Total de páginas encontradas: 43
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=1
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=2
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=3
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=4
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=5
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=6
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=7
Processando a página: https://www.casarossi.com.br/pisos-e-revestimentos?pg=8
Processando a página: https://www.casarossi.com.br/pisos-e-re